# **DATA CLEANING**

Link: https://www.kaggle.com/datasets/yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018?resource=download&select=2017.csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1. Inspecting Data

We load the data,

In [2]:
df = pd.read_csv("2018.csv")
df.head()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27
0,2018-01-01,UA,2429,EWR,DEN,1517,1512.0,-5.0,15.0,1527.0,...,268.0,250.0,225.0,1605.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-01,UA,2427,LAS,SFO,1115,1107.0,-8.0,11.0,1118.0,...,99.0,83.0,65.0,414.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-01,UA,2426,SNA,DEN,1335,1330.0,-5.0,15.0,1345.0,...,134.0,126.0,106.0,846.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-01,UA,2425,RSW,ORD,1546,1552.0,6.0,19.0,1611.0,...,190.0,182.0,157.0,1120.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-01,UA,2424,ORD,ALB,630,650.0,20.0,13.0,703.0,...,112.0,106.0,83.0,723.0,NaN,NaN,NaN,NaN,NaN,NaN


We want the column names and their data types,

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7213446 entries, 0 to 7213445
Data columns (total 28 columns):
 #   Column               Dtype  
---  ------               -----  
 0   FL_DATE              str    
 1   OP_CARRIER           str    
 2   OP_CARRIER_FL_NUM    int64  
 3   ORIGIN               str    
 4   DEST                 str    
 5   CRS_DEP_TIME         int64  
 6   DEP_TIME             float64
 7   DEP_DELAY            float64
 8   TAXI_OUT             float64
 9   WHEELS_OFF           float64
 10  WHEELS_ON            float64
 11  TAXI_IN              float64
 12  CRS_ARR_TIME         int64  
 13  ARR_TIME             float64
 14  ARR_DELAY            float64
 15  CANCELLED            float64
 16  CANCELLATION_CODE    str    
 17  DIVERTED             float64
 18  CRS_ELAPSED_TIME     float64
 19  ACTUAL_ELAPSED_TIME  float64
 20  AIR_TIME             float64
 21  DISTANCE             float64
 22  CARRIER_DELAY        float64
 23  WEATHER_DELAY        float64
 24  NAS_DELAY

Some data types appear to need changing, because they are inappropriate. For example, we may consider changing `FL_DATE` from `str` to `datetime`. Furthermore, there also seems to be numerous columns which have many empty entries. Let us investigate this,

In [4]:
df.isnull().sum()

FL_DATE                      0
OP_CARRIER                   0
OP_CARRIER_FL_NUM            0
ORIGIN                       0
DEST                         0
CRS_DEP_TIME                 0
DEP_TIME                112317
DEP_DELAY               117234
TAXI_OUT                115830
WHEELS_OFF              115829
WHEELS_ON               119246
TAXI_IN                 119246
CRS_ARR_TIME                 0
ARR_TIME                119245
ARR_DELAY               137040
CANCELLED                    0
CANCELLATION_CODE      7096862
DIVERTED                     0
CRS_ELAPSED_TIME            10
ACTUAL_ELAPSED_TIME     134442
AIR_TIME                134442
DISTANCE                     0
CARRIER_DELAY          5860736
WEATHER_DELAY          5860736
NAS_DELAY              5860736
SECURITY_DELAY         5860736
LATE_AIRCRAFT_DELAY    5860736
Unnamed: 27            7213446
dtype: int64

We should remove the columns which have too much missing information. These columns include `CANCELLATION_CODE`, `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY` and `Unnamed: 27`. It is also a good idea to check for any duplicate rows,

In [5]:
df.duplicated().sum()

np.int64(0)

Luckily for us, there are no duplicate rows that we need to get rid of.

### 2. Removing Columns

We start by removing the problematic columns that have too many missing entries,

In [6]:
df_eng = df.drop(columns=["CANCELLATION_CODE", 
                              "CARRIER_DELAY", 
                              "WEATHER_DELAY", 
                              "NAS_DELAY", 
                              "SECURITY_DELAY", 
                              "LATE_AIRCRAFT_DELAY", 
                              "Unnamed: 27"])

Now we have,

In [7]:
df_eng.info()

<class 'pandas.DataFrame'>
RangeIndex: 7213446 entries, 0 to 7213445
Data columns (total 21 columns):
 #   Column               Dtype  
---  ------               -----  
 0   FL_DATE              str    
 1   OP_CARRIER           str    
 2   OP_CARRIER_FL_NUM    int64  
 3   ORIGIN               str    
 4   DEST                 str    
 5   CRS_DEP_TIME         int64  
 6   DEP_TIME             float64
 7   DEP_DELAY            float64
 8   TAXI_OUT             float64
 9   WHEELS_OFF           float64
 10  WHEELS_ON            float64
 11  TAXI_IN              float64
 12  CRS_ARR_TIME         int64  
 13  ARR_TIME             float64
 14  ARR_DELAY            float64
 15  CANCELLED            float64
 16  DIVERTED             float64
 17  CRS_ELAPSED_TIME     float64
 18  ACTUAL_ELAPSED_TIME  float64
 19  AIR_TIME             float64
 20  DISTANCE             float64
dtypes: float64(14), int64(3), str(4)
memory usage: 1.1 GB


### 3. Data Types

The next step is to sort out the data types,

In [8]:
# Defining functions,
def convert_to_datetime(df, columns):
    """Function to convert a time column into the datetime format HH:MM."""
    for column in columns:
        df[column] = pd.to_datetime(df[column].astype("Int64"), format="%H%M", errors="coerce").dt.strftime("%H:%M")

def convert_float_to_int(df, columns):
    """We convert all float columns into the integer format."""
    for column in columns:
        df[column] = df[column].astype("Int64")

# Converting,
convert_to_datetime(df_eng, 
                    ["CRS_DEP_TIME", 
                     "DEP_TIME",
                     "WHEELS_OFF",
                     "WHEELS_ON",
                     "CRS_ARR_TIME", 
                     "ARR_TIME"])

convert_float_to_int(df_eng,
                    ["DEP_DELAY", 
                     "TAXI_OUT", 
                     "TAXI_IN",
                     "ARR_DELAY",
                     "CANCELLED",
                     "DIVERTED",
                     "CRS_ELAPSED_TIME",
                     "ACTUAL_ELAPSED_TIME",
                     "AIR_TIME",
                     "DISTANCE"])

Our dataframe now,

In [9]:
df_eng.info()

<class 'pandas.DataFrame'>
RangeIndex: 7213446 entries, 0 to 7213445
Data columns (total 21 columns):
 #   Column               Dtype
---  ------               -----
 0   FL_DATE              str  
 1   OP_CARRIER           str  
 2   OP_CARRIER_FL_NUM    int64
 3   ORIGIN               str  
 4   DEST                 str  
 5   CRS_DEP_TIME         str  
 6   DEP_TIME             str  
 7   DEP_DELAY            Int64
 8   TAXI_OUT             Int64
 9   WHEELS_OFF           str  
 10  WHEELS_ON            str  
 11  TAXI_IN              Int64
 12  CRS_ARR_TIME         str  
 13  ARR_TIME             str  
 14  ARR_DELAY            Int64
 15  CANCELLED            Int64
 16  DIVERTED             Int64
 17  CRS_ELAPSED_TIME     Int64
 18  ACTUAL_ELAPSED_TIME  Int64
 19  AIR_TIME             Int64
 20  DISTANCE             Int64
dtypes: Int64(10), int64(1), str(10)
memory usage: 1.2 GB


### 4. Aggressive Cleaning

Flights which are cancelled or diverted have many missing entries such as departure time, arrival time, enlapsed time, etc. because the plane never took off and/or arrived at its intended destination. However, we want to make sure that we do not have any missing depature and arrival information for flights that were not cancelled or diverted. We can see that we have many of these cases, 

In [10]:
df_eng[(df_eng["CANCELLED"] == 0) & (df_eng["DIVERTED"] == 0)].isnull().sum()

FL_DATE                    0
OP_CARRIER                 0
OP_CARRIER_FL_NUM          0
ORIGIN                     0
DEST                       0
CRS_DEP_TIME            1303
DEP_TIME                6336
DEP_DELAY               4725
TAXI_OUT                   0
WHEELS_OFF             10649
WHEELS_ON              30327
TAXI_IN                    0
CRS_ARR_TIME           22761
ARR_TIME               34863
ARR_DELAY               2599
CANCELLED                  0
DIVERTED                   0
CRS_ELAPSED_TIME           0
ACTUAL_ELAPSED_TIME        1
AIR_TIME                   1
DISTANCE                   0
dtype: int64

Removing these columns,

In [11]:
df_eng.drop(df_eng.index[
    (df_eng["CANCELLED"] == 0) &
    (df_eng["DIVERTED"] == 0) &
    df_eng.drop(columns=["CANCELLED", "DIVERTED"]).isna().any(axis=1)
    ],inplace=True
)

Our dataframe is now cleaned,

In [12]:
df_eng

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE
0,2018-01-01,UA,2429,EWR,DEN,15:17,15:12,-5,15,15:27,...,10,17:45,17:22,-23,0,0,268,250,225,1605
1,2018-01-01,UA,2427,LAS,SFO,11:15,11:07,-8,11,11:18,...,7,12:54,12:30,-24,0,0,99,83,65,414
2,2018-01-01,UA,2426,SNA,DEN,13:35,13:30,-5,15,13:45,...,5,16:49,16:36,-13,0,0,134,126,106,846
3,2018-01-01,UA,2425,RSW,ORD,15:46,15:52,6,19,16:11,...,6,17:56,17:54,-2,0,0,190,182,157,1120
4,2018-01-01,UA,2424,ORD,ALB,06:30,06:50,20,13,07:03,...,10,09:22,09:36,14,0,0,112,106,83,723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7213441,2018-12-31,AA,1815,DCA,CLT,15:34,15:30,-4,20,15:50,...,7,17:14,17:09,-5,0,0,100,99,72,331
7213442,2018-12-31,AA,1816,CLT,DFW,17:51,17:57,6,18,18:15,...,10,19:52,19:53,1,0,0,181,176,148,936
7213443,2018-12-31,AA,1817,CLT,MEM,20:15,20:10,-5,36,20:46,...,4,21:07,21:18,11,0,0,112,128,88,511
7213444,2018-12-31,AA,1818,CLT,RDU,13:00,13:23,23,11,13:34,...,4,13:50,14:04,14,0,0,50,41,26,130


### 5. Feature Engineering

We add an additional column called `TOTAL_DELAY` which is simply the difference between the expected and actual enlapsed times `CRS_ELAPSED_TIME` and `ACTUAL_ELAPSED_TIME` respectively.

In [13]:
df_eng["TOTAL_DELAY"] = df_eng["ACTUAL_ELAPSED_TIME"] - df_eng["CRS_ELAPSED_TIME"]

### 6. Creating The Local Database

If we do not have the database already created, we must construct an engine from `sqlalchemy` to create it,

In [16]:
# Importing,
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer

# Creating the engine,
engine = create_engine("sqlite:///flight_data.db", echo=True)
metadata = MetaData()

# Creating table,
flights = Table(
    "flights",
    metadata,
    Column("FL_DATE", String),
    Column("OP_CARRIER", String),
    Column("OP_CARRIER_FL_NUM", Integer),
    Column("ORIGIN", String),
    Column("DEST", String),
    Column("CRS_DEP_TIME", String),
    Column("DEP_TIME", String),
    Column("DEP_DELAY", Integer),
    Column("TAXI_OUT", Integer),
    Column("WHEELS_OFF", String),
    Column("WHEELS_ON", String),
    Column("TAXI_IN", Integer),
    Column("CRS_ARR_TIME", String),
    Column("ARR_TIME", String),
    Column("ARR_DELAY", Integer),
    Column("CANCELLED", Integer),
    Column("DIVERTED", Integer),
    Column("CRS_ELAPSED_TIME", Integer),
    Column("ACTUAL_ELAPSED_TIME", Integer),
    Column("AIR_TIME", Integer),
    Column("DISTANCE", Integer),
    Column("TOTAL_DELAY", Integer)
)

metadata.create_all(engine)

2026-11-18 12:23:48,393 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-11-18 12:23:48,394 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("flights")
2026-11-18 12:23:48,395 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-11-18 12:23:48,398 INFO sqlalchemy.engine.Engine COMMIT


Now we dump the rows from our dataframe into our local database,

In [18]:
df_eng.to_sql(
    name="flights",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

2026-11-18 12:27:51,916 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-11-18 12:27:51,938 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("flights")
2026-11-18 12:27:51,938 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-11-18 12:28:04,414 INFO sqlalchemy.engine.Engine INSERT INTO flights ("FL_DATE", "OP_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY", "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN", "CRS_ARR_TIME", "ARR_TIME", "ARR_DELAY", "CANCELLED", "DIVERTED", "CRS_ELAPSED_TIME", "ACTUAL_ELAPSED_TIME", "AIR_TIME", "DISTANCE", "TOTAL_DELAY") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
2026-11-18 12:28:04,488 INFO sqlalchemy.engine.Engine [generated in 0.13386s] [('2018-01-01', 'UA', 2429, 'EWR', 'DEN', '15:17', '15:12', -5, 15, '15:27', '17:12', 10, '17:45', '17:22', -23, 0, 0, 268, 250, 225, 1605, -18), ('2018-01-01', 'UA', 2427, 'LAS', 'SFO', '11:15', '11:07', -8, 11, '11:18', '12:23', 7, '12:54', '

7120314